<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/Ciencia%20De%20Datos%20Con%20Python/CuadernosColab/CienciaDeDatos_Cap%C3%ADtulo018_Conversi%C3%B3n_de_tipos_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 18 — Conversión de tipos de datos

## Breve repaso

En el trabajo anterior sobre limpieza de textos y categorías vimos que no alcanza con mirar el contenido de una columna de manera superficial. Dos valores pueden parecer equivalentes para una persona, pero ser distintos para Pandas si tienen espacios, mayúsculas, tildes o variantes de escritura.

En este capítulo vamos a trabajar con otro aspecto fundamental de la preparación de datos: los tipos de datos.

Cuando Pandas carga un archivo, intenta interpretar automáticamente qué tipo de información contiene cada columna. Algunas columnas quedan como números, otras como texto y otras pueden necesitar una conversión especial. Sin embargo, esa interpretación automática no siempre coincide con lo que necesitamos para analizar.

Una columna puede “parecer” numérica porque contiene valores como `2`, `3.5` o `1200`, pero si está cargada como texto, Pandas no la tratará como una variable numérica real. Algo similar ocurre con las fechas: una fecha puede verse como `2024-03-01`, pero si Pandas la interpreta como texto, no podremos trabajarla correctamente como dato temporal.

En este capítulo vamos a revisar tipos de datos, detectar columnas que necesitan conversión y transformar valores usando herramientas como `pd.to_numeric()` y `pd.to_datetime()`.

Al finalizar este capítulo deberías poder:

- Comprender por qué los tipos de datos son importantes para el análisis.
- Revisar los tipos de columnas con `dtypes` e `info()`.
- Detectar columnas numéricas cargadas como texto.
- Convertir columnas a formato numérico usando `pd.to_numeric()`.
- Comprender el uso de `errors="coerce"`.
- Verificar si la conversión generó nuevos valores faltantes.
- Convertir una columna de texto a fecha usando `pd.to_datetime()`.
- Validar la estructura del `DataFrame` después de convertir tipos.

## Dataset de trabajo

Para estudiar conversión de tipos de datos vamos a volver al dataset real **Cafe Sales — Dirty Data for Cleaning Training**.

Este dataset contiene transacciones de venta de una cafetería. Algunas columnas deberían poder analizarse como valores numéricos, por ejemplo `Quantity`, `Price Per Unit` y `Total Spent`. También hay una columna de fecha, `Transaction Date`, que debería poder interpretarse como dato temporal.

Sin embargo, como el dataset contiene valores problemáticos, Pandas puede cargar algunas de estas columnas con tipos de datos que no son los más adecuados para el análisis.

Como cada capítulo debe poder ejecutarse de manera independiente, vamos a descargar y cargar nuevamente el dataset.

In [47]:
import kagglehub
import pandas as pd
import os

ruta_dataset = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

archivos = os.listdir(ruta_dataset)

archivos_csv = [
    archivo for archivo in archivos
    if archivo.endswith(".csv")
]

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])

df = pd.read_csv(ruta_csv)

df.head()

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


La salida de `head()` nos permite confirmar que el archivo fue cargado correctamente.

En este capítulo no vamos a concentrarnos todavía en limpiar todos los problemas del dataset. El foco estará puesto en revisar cómo Pandas interpretó los tipos de datos y en convertir algunas columnas para que puedan usarse correctamente en cálculos y análisis posteriores.

## Revisar los tipos de datos

Después de cargar un dataset, una de las primeras tareas de diagnóstico consiste en revisar qué tipo de dato asignó Pandas a cada columna.

Esto es importante porque muchas operaciones dependen del tipo de dato. Si una columna fue interpretada como numérica, podremos calcular promedios, medianas, sumas o rankings. Si una columna fue interpretada como texto, esas operaciones pueden fallar o producir resultados que no tienen sentido.

Podemos revisar los tipos de datos con `dtypes`.

In [48]:
df.dtypes

,0
Transaction ID,object
Item,object
Quantity,object
Price Per Unit,object
Total Spent,object
Payment Method,object
Location,object
Transaction Date,object


La salida muestra el tipo de dato de cada columna.

En Pandas, el tipo `object` suele indicar texto o valores mezclados. No siempre significa que la columna esté mal, pero sí nos invita a revisar con cuidado.

En este dataset, columnas como `Item`, `Payment Method` o `Location` pueden ser textuales. Eso es esperable. Pero si columnas como `Quantity`, `Price Per Unit` o `Total Spent` aparecen como `object`, tenemos una señal de problema: son columnas que deberían poder analizarse como numéricas.

También podemos usar `info()` para obtener una visión general de tipos y valores no nulos.

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


La salida de `info()` permite ver, en una misma tabla, el nombre de cada columna, la cantidad de valores no nulos y el tipo de dato asignado por Pandas.

Esta revisión es parte del diagnóstico inicial. Antes de convertir columnas, necesitamos identificar cuáles parecen tener un tipo de dato inadecuado y por qué eso puede afectar el análisis.

## Interpretar la salida de info()

La salida de `info()` nos muestra algo importante: todas las columnas del dataset fueron cargadas con tipo `object`.

Esto significa que Pandas está tratando todas las columnas como texto o como columnas con valores mezclados.

En algunas columnas esto es esperable. Por ejemplo, `Transaction ID` es un identificador, por lo tanto no necesitamos tratarlo como número. Lo mismo ocurre con `Item`, `Payment Method` y `Location`, que representan categorías o textos.

Pero hay otras columnas que deberían poder analizarse de otra manera:

```text
Quantity           → debería ser numérica
Price Per Unit     → debería ser numérica
Total Spent        → debería ser numérica
Transaction Date   → debería ser una fecha
```

Si `Quantity`, `Price Per Unit` y `Total Spent` permanecen como `object`, no podremos trabajar con ellas de manera confiable como columnas numéricas. Esto puede afectar cálculos como sumas, promedios, medianas, rankings o validaciones entre columnas.

Algo similar ocurre con `Transaction Date`. Aunque sus valores parezcan fechas, mientras Pandas la interprete como `object`, esa columna sigue siendo texto. Para analizarla temporalmente, ordenar fechas, extraer meses o agrupar ventas por día, vamos a necesitar convertirla a un tipo de dato temporal.

También vemos que varias columnas tienen menos de 10000 valores no nulos. Eso indica que hay valores faltantes. En este capítulo no vamos a centrarnos en el tratamiento de faltantes, porque ya lo trabajamos antes. Sin embargo, al convertir tipos debemos tenerlos en cuenta, porque algunos errores de conversión también pueden generar nuevos valores faltantes.

Antes de analizar este dataset, necesitamos convertir algunas columnas al tipo de dato adecuado.


## Observar columnas que deberían ser numéricas

Antes de convertir columnas, conviene observar algunos valores.

En este dataset hay tres columnas que deberían representar cantidades o importes:

```text
Quantity
Price Per Unit
Total Spent
```

Aunque visualmente puedan parecer numéricas, Pandas las cargó como `object`. Eso suele ocurrir cuando una columna contiene números mezclados con textos, valores problemáticos o datos faltantes.

Vamos a observar algunos valores de esas columnas.

In [50]:
df[["Quantity", "Price Per Unit", "Total Spent"]].head(15)

,Quantity,Price Per Unit,Total Spent
0,2,2.0,4.0
1,4,3.0,12.0
2,4,1.0,ERROR
3,2,5.0,10.0
4,2,2.0,4.0
5,5,4.0,20.0
6,3,3.0,9.0
7,4,4.0,16.0
8,5,3.0,15.0
9,5,4.0,20.0


Esta vista nos permite revisar ejemplos concretos.

Si una columna contiene valores como `"ERROR"` o `"UNKNOWN"`, Pandas no puede convertirla automáticamente a número durante la carga del archivo. Por eso la conserva como `object`.

También podemos revisar las frecuencias de valores en cada una de estas columnas. Esto puede ayudarnos a detectar valores problemáticos.

In [51]:
df["Quantity"].value_counts(dropna=False).head(15)

,count
Quantity,
5,2013
2,1974
4,1863
3,1849
1,1822
UNKNOWN,171
ERROR,170
NaN,138


In [52]:
df["Price Per Unit"].value_counts(dropna=False).head(15)

,count
Price Per Unit,
3.0,2429
4.0,2331
2.0,1227
5.0,1204
1.0,1143
1.5,1133
ERROR,190
NaN,179
UNKNOWN,164


In [53]:
df["Total Spent"].value_counts(dropna=False).head(15)

,count
Total Spent,
6.0,979
12.0,939
3.0,930
4.0,923
20.0,746
15.0,734
8.0,677
10.0,524
2.0,497


En estas salidas debemos prestar atención a valores que no sean números reales.

Si aparecen valores como `"UNKNOWN"` o `"ERROR"`, ya tenemos una explicación para el tipo `object`: la columna mezcla valores numéricos con textos problemáticos.

Antes de convertir, es importante detectar estos casos. Convertir tipos no debería ser una acción ciega. Necesitamos saber qué clase de valores contiene la columna y qué ocurrirá con aquellos que no puedan convertirse.

## Por qué no alcanza con que los valores parezcan números

Una columna puede verse numérica cuando observamos algunas filas, pero eso no garantiza que Pandas pueda tratarla como una columna numérica.

Por ejemplo, `Quantity`, `Price Per Unit` y `Total Spent` representan valores que deberían permitir cálculos. Sin embargo, como fueron cargadas como `object`, necesitamos revisar si pueden usarse directamente en operaciones numéricas.

Intentemos calcular algunas estadísticas sobre `Price Per Unit`.

In [54]:
df["Price Per Unit"].describe()

,Price Per Unit
count,9821
unique,8
top,3.0
freq,2429


Como la columna fue cargada como `object`, `describe()` produce un resumen propio de datos categóricos o textuales, no un resumen numérico.

En lugar de mostrar medidas como media, mediana, mínimo o máximo, Pandas muestra información como cantidad de valores, cantidad de valores únicos, valor más frecuente y frecuencia del valor más frecuente.

Esto nos indica que, para Pandas, `Price Per Unit` todavía no es una variable numérica.

Algo similar ocurre si intentamos ordenar o comparar valores. Cuando una columna está como texto, el orden puede ser alfabético y no necesariamente numérico. Por eso, antes de hacer análisis cuantitativo, necesitamos convertir estas columnas al tipo adecuado.

In [55]:
df["Price Per Unit"].sort_values().head(10)

,Price Per Unit
1207,1.0
5841,1.0
2606,1.0
3933,1.0
1331,1.0
5834,1.0
3935,1.0
5832,1.0
3942,1.0
8544,1.0


Esta salida puede resultar engañosa, porque el ordenamiento se realiza sobre textos.

En análisis de datos, no alcanza con que una columna parezca contener números. Necesitamos que Pandas la reconozca como numérica para poder calcular, comparar y ordenar correctamente.

El siguiente paso será convertir estas columnas usando `pd.to_numeric()`.

## Convertir columnas con pd.to_numeric()

Para convertir una columna a formato numérico podemos usar `pd.to_numeric()`.

Esta función intenta transformar los valores de una columna en números. Si todos los valores son compatibles con un formato numérico, la conversión suele ser directa.

Probemos primero con la columna `Price Per Unit`.

In [56]:
# pd.to_numeric(df["Price Per Unit"])

Es posible que esta instrucción produzca un error.

Ese error aparece porque la columna no contiene solamente números. También contiene valores textuales problemáticos, como `"ERROR"` o `"UNKNOWN"`. Pandas no puede convertir esos textos en números, porque no representan cantidades válidas.

Este resultado es importante: confirma que el problema no era solamente el tipo `object`, sino la presencia de valores no numéricos dentro de una columna que debería ser numérica.

Para manejar este tipo de situación podemos usar el parámetro `errors`.

## Controlar errores de conversión

Cuando usamos `pd.to_numeric()`, podemos indicar qué debe hacer Pandas si encuentra un valor que no puede convertir.

Para eso usamos el parámetro `errors`.

Una opción muy útil es `errors="coerce"`.

Con esta opción, Pandas intenta convertir cada valor a número. Si encuentra un valor que no puede convertir, como `"ERROR"` o `"UNKNOWN"`, lo transforma en `NaN`.

Esto no significa que el problema desaparezca. Significa que los valores no convertibles quedan marcados como faltantes, lo cual nos permite detectarlos y tratarlos con las herramientas que ya conocemos.


In [57]:
precio_convertido = pd.to_numeric(
    df["Price Per Unit"],
    errors="coerce"
)

precio_convertido.head(15)

,Price Per Unit
0,2.0
1,3.0
2,1.0
3,5.0
4,2.0
5,4.0
6,3.0
7,4.0
8,3.0
9,4.0


Ahora la conversión no se detiene con un error.

Los valores numéricos se transforman correctamente, y los valores que no pudieron convertirse pasan a ser `NaN`.

Podemos revisar el tipo de dato resultante:

In [58]:
precio_convertido.dtype

dtype('float64')

El resultado ya no es `object`. Ahora tenemos una serie numérica.

También podemos revisar cuántos valores faltantes aparecen después de la conversión:

In [59]:
precio_convertido.isna().sum()

np.int64(533)

Este conteo incluye los valores que ya eran faltantes y también los valores que se convirtieron en `NaN` porque no pudieron interpretarse como números.

Por eso, después de usar `errors="coerce"`, siempre debemos verificar cuántos valores quedaron como faltantes. La conversión no debe ocultar el problema: debe hacerlo más fácil de diagnosticar.

## Convertir varias columnas numéricas

En nuestro dataset no necesitamos convertir una sola columna. Hay tres columnas que deberían poder tratarse como numéricas:

```text
Quantity
Price Per Unit
Total Spent
```

Como no queremos modificar el dataset original directamente, vamos a crear una copia llamada `df_convertido`.

Luego aplicaremos `pd.to_numeric()` a cada una de esas columnas.


In [60]:
df_convertido = df.copy()

columnas_numericas = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

for columna in columnas_numericas:
    df_convertido[columna] = pd.to_numeric(
        df_convertido[columna],
        errors="coerce"
    )

df_convertido[columnas_numericas].head()

,Quantity,Price Per Unit,Total Spent
0,2.0,2.0,4.0
1,4.0,3.0,12.0
2,4.0,1.0,NaN
3,2.0,5.0,10.0
4,2.0,2.0,4.0


Ahora las columnas seleccionadas fueron convertidas a formato numérico dentro de `df_convertido`.

Usamos un bucle `for` para aplicar la misma transformación a varias columnas. Esto evita repetir tres veces una estructura muy parecida.

La lógica fue la misma en cada caso: intentar convertir los valores a número y transformar en `NaN` aquellos valores que no podían interpretarse como numéricos.

Después de convertir, debemos verificar los tipos de datos.

In [61]:
df_convertido[columnas_numericas].dtypes

,0
Quantity,float64
Price Per Unit,float64
Total Spent,float64


Ahora estas columnas ya no aparecen como `object`. Pandas puede tratarlas como columnas numéricas.

Esto nos permite calcular estadísticas, ordenar correctamente, comparar valores y construir nuevas columnas a partir de operaciones matemáticas.

Sin embargo, todavía debemos revisar algo importante: al usar `errors="coerce"`, algunos valores problemáticos pueden haberse convertido en `NaN`. Por eso, después de convertir, siempre necesitamos verificar los faltantes.

## Verificar faltantes después de convertir

Cuando usamos `pd.to_numeric()` con `errors="coerce"`, los valores que no pueden convertirse pasan a ser `NaN`.

Eso significa que la conversión puede aumentar la cantidad de valores faltantes detectados por Pandas. No porque hayamos perdido datos, sino porque valores como `"ERROR"` o `"UNKNOWN"` fueron transformados en faltantes reconocidos.

Por eso, conviene comparar los faltantes antes y después de la conversión.

In [62]:
faltantes_antes = df[columnas_numericas].isna().sum()
faltantes_despues = df_convertido[columnas_numericas].isna().sum()

comparacion_faltantes = pd.DataFrame({
    "faltantes_antes": faltantes_antes,
    "faltantes_despues": faltantes_despues,
    "nuevos_faltantes": faltantes_despues - faltantes_antes
})

comparacion_faltantes

,faltantes_antes,faltantes_despues,nuevos_faltantes
Quantity,138,479,341
Price Per Unit,179,533,354
Total Spent,173,502,329


La columna `faltantes_antes` muestra los valores que Pandas ya reconocía como faltantes antes de la conversión.

La columna `faltantes_despues` muestra los faltantes luego de convertir las columnas a formato numérico.

La columna `nuevos_faltantes` indica cuántos valores adicionales pasaron a ser `NaN` porque no pudieron convertirse.

Esta comparación es muy importante. Si solo miramos el resultado final, podríamos pensar que la columna simplemente tenía muchos faltantes. Pero al comparar antes y después vemos que una parte de esos faltantes proviene de valores problemáticos que estaban escritos como texto.

La conversión de tipos no solo cambia el formato de una columna. También puede revelar problemas de calidad que antes estaban ocultos.

## Calcular estadísticas después de convertir

Una vez convertidas las columnas numéricas, Pandas puede analizarlas como cantidades reales.

Antes de la conversión, `describe()` sobre `Price Per Unit` producía un resumen de tipo textual. Ahora, sobre `df_convertido`, podemos obtener estadísticas numéricas.

In [63]:
df_convertido["Price Per Unit"].describe()

,Price Per Unit
count,9467.000000
mean,2.949984
std,1.278450
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000


Ahora la salida incluye medidas como media, desviación estándar, mínimo, cuartiles y máximo.

Esto confirma que la columna ya puede usarse como una variable numérica.

También podemos obtener un resumen de las tres columnas convertidas:

In [64]:
df_convertido[columnas_numericas].describe()

,Quantity,Price Per Unit,Total Spent
count,9521.000000,9467.000000,9498.000000
mean,3.028463,2.949984,8.924352
std,1.419007,1.278450,6.009919
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000


Este resumen permite revisar rangos y valores generales de las columnas numéricas.

A partir de esta conversión, podemos hacer operaciones que antes no eran confiables. Por ejemplo, podemos ordenar por precio unitario de mayor a menor:

In [65]:
df_convertido.sort_values("Price Per Unit", ascending=False).head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9958,TXN_4125474,ERROR,2.0,5.0,10.0,Credit Card,In-store,2023-08-02
9942,TXN_5344848,Salad,1.0,5.0,5.0,Digital Wallet,Takeaway,2023-09-27
9940,TXN_8273780,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-10-15
9928,TXN_5171519,Salad,3.0,5.0,15.0,NaN,In-store,2023-04-06
9920,TXN_4365321,Salad,4.0,5.0,20.0,Cash,In-store,2023-03-09


También podemos crear o verificar relaciones entre columnas numéricas.

Por ejemplo, más adelante podremos revisar si `Total Spent` coincide con `Quantity * Price Per Unit`.

La conversión de tipos no es solo un cambio técnico. Es lo que permite que ciertas columnas puedan participar correctamente en cálculos, comparaciones, ordenamientos y validaciones.

## Convertir una columna de fecha

Además de columnas numéricas, muchos datasets incluyen columnas de fecha.

En nuestro caso, la columna `Transaction Date` representa la fecha de cada transacción. Sin embargo, al cargar el archivo, Pandas la interpretó como `object`. Eso significa que, por ahora, la está tratando como texto.

Podemos verificarlo:

In [66]:
df_convertido["Transaction Date"].dtype

dtype('O')

Aunque los valores se vean como fechas, todavía no son datos temporales para Pandas.

Para convertir una columna a formato de fecha usamos `pd.to_datetime()`.

In [67]:
fecha_convertida = pd.to_datetime(
    df_convertido["Transaction Date"],
    errors="coerce"
)

fecha_convertida.head()

,Transaction Date
0,2023-09-08
1,2023-05-16
2,2023-07-19
3,2023-04-27
4,2023-06-11


La función `pd.to_datetime()` intenta interpretar los valores como fechas.

Usamos nuevamente `errors="coerce"` para evitar que la conversión se detenga si aparece un valor que no puede interpretarse como fecha. En esos casos, Pandas convierte el valor problemático en `NaT`.

`NaT` significa *Not a Time* y es el equivalente temporal de un valor faltante. Así como `NaN` representa un faltante en muchos tipos de columnas, `NaT` representa una fecha faltante o no interpretable.

Ahora podemos revisar el tipo de dato resultante:

In [68]:
fecha_convertida.dtype

dtype('<M8[ns]')

El tipo `datetime64[ns]` indica que Pandas ya interpreta la columna como una fecha.

Podemos guardar esta conversión en el `DataFrame` convertido:

In [69]:
df_convertido["Transaction Date"] = pd.to_datetime(
    df_convertido["Transaction Date"],
    errors="coerce"
)

df_convertido["Transaction Date"].dtype

dtype('<M8[ns]')

A partir de esta conversión, la columna `Transaction Date` puede usarse como dato temporal.

Todavía no vamos a profundizar en operaciones con fechas. Más adelante podremos extraer el año, el mes, el día, ordenar correctamente las transacciones en el tiempo o agrupar ventas por período.

En este capítulo nos alcanza con comprender la idea principal: una fecha escrita como texto no es lo mismo que una fecha interpretada como dato temporal por Pandas.

## Verificar la conversión de fechas

Así como hicimos con las columnas numéricas, después de convertir una columna de fecha conviene verificar el resultado.

Primero podemos revisar cuántos valores faltantes tenía originalmente la columna `Transaction Date` y cuántos valores faltantes o no interpretables aparecen después de la conversión.

In [70]:
faltantes_fecha_antes = df["Transaction Date"].isna().sum()
faltantes_fecha_despues = df_convertido["Transaction Date"].isna().sum()

print("Faltantes antes de convertir:")
print(faltantes_fecha_antes)

print()

print("Faltantes después de convertir:")
print(faltantes_fecha_despues)

Faltantes antes de convertir:
159

Faltantes después de convertir:
460


Si la cantidad de faltantes aumenta después de la conversión, eso significa que algunos valores que antes estaban escritos como texto no pudieron interpretarse como fechas y fueron convertidos en `NaT`.

También podemos revisar algunas estadísticas básicas de la columna convertida.

In [71]:
df_convertido["Transaction Date"].describe()

,Transaction Date
count,9540
mean,2023-07-01 23:00:31.698113280
min,2023-01-01 00:00:00
25%,2023-04-01 00:00:00
50%,2023-07-02 00:00:00
75%,2023-10-02 00:00:00
max,2023-12-31 00:00:00


El resumen de una columna temporal nos permite observar, entre otras cosas, la fecha mínima y la fecha máxima registradas.

Esto puede ser útil para detectar fechas fuera de rango o valores extraños. Por ejemplo, si un dataset de ventas de 2024 tuviera una fecha del año 2099, esa fecha debería llamar nuestra atención.

También podemos ordenar el dataset por fecha:

In [72]:
df_convertido.sort_values("Transaction Date").head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
8015,TXN_4801947,Juice,1.0,3.0,3.0,Digital Wallet,Takeaway,2023-01-01
9063,TXN_9161256,Smoothie,2.0,4.0,8.0,Digital Wallet,In-store,2023-01-01
7309,TXN_6093955,Tea,5.0,1.5,NaN,UNKNOWN,Takeaway,2023-01-01
1425,TXN_8842223,Sandwich,5.0,NaN,20.0,Digital Wallet,In-store,2023-01-01
1777,TXN_7367474,Juice,5.0,3.0,15.0,Digital Wallet,Takeaway,2023-01-01


Ahora el ordenamiento se realiza con una columna temporal, no con texto.

En muchos casos esto no cambia visualmente el resultado si las fechas estaban escritas en un formato ordenable, como `YYYY-MM-DD`. Pero conceptualmente sí es importante: Pandas ahora reconoce la columna como fecha y puede aplicar operaciones temporales sobre ella.

La conversión de fechas será la base para trabajar más adelante con análisis por día, mes, año o períodos.

## Validar los tipos finales

Después de convertir columnas numéricas y temporales, conviene revisar nuevamente la estructura general del `DataFrame`.

El objetivo es comprobar que las columnas que necesitaban conversión ya tengan tipos de datos más adecuados para el análisis.

In [73]:
df_convertido.dtypes

,0
Transaction ID,object
Item,object
Quantity,float64
Price Per Unit,float64
Total Spent,float64
Payment Method,object
Location,object
Transaction Date,datetime64[ns]


Ahora deberíamos observar que:

```text
Quantity           → tipo numérico
Price Per Unit     → tipo numérico
Total Spent        → tipo numérico
Transaction Date   → tipo fecha
```

Mientras tanto, columnas como `Transaction ID`, `Item`, `Payment Method` y `Location` pueden seguir como `object`, porque representan identificadores o categorías textuales.

También podemos usar `info()` para ver la estructura completa:

In [74]:
df_convertido.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9667 non-null   object        
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    7421 non-null   object        
 6   Location          6735 non-null   object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB


Esta revisión final permite confirmar que el dataset está en una mejor condición para el análisis.

Convertir tipos no significa que el dataset ya esté completamente limpio. Todavía pueden quedar valores faltantes, categorías problemáticas, duplicados u otras inconsistencias. Pero sí significa que algunas columnas ya están en un formato más apropiado para operar con ellas.

A partir de este momento, las columnas numéricas pueden participar en cálculos y comparaciones, y la columna de fecha puede usarse para análisis temporal.

La validación final es una parte esencial del proceso. No basta con aplicar conversiones: necesitamos comprobar que el resultado tenga sentido.

## Errores frecuentes al convertir tipos de datos

Al convertir tipos de datos, uno de los errores más comunes es asumir que una columna está bien solo porque sus valores “parecen” numéricos o “parecen” fechas.

Para Pandas, no alcanza con la apariencia visual. Si una columna fue cargada como `object`, puede contener textos, valores problemáticos o datos mezclados. Antes de usarla en cálculos, conviene revisar su tipo y convertirla si corresponde.

Otro error frecuente es usar `pd.to_numeric()` o `pd.to_datetime()` sin verificar qué ocurrió después. Cuando usamos `errors="coerce"`, los valores que no pueden convertirse pasan a ser `NaN` o `NaT`. Eso es útil porque evita que la conversión se detenga, pero también puede aumentar la cantidad de valores faltantes.

Por eso, después de convertir, siempre conviene comparar faltantes antes y después.

También debemos evitar sobrescribir el dataset original demasiado pronto. En este capítulo trabajamos con una copia llamada `df_convertido`, lo que nos permitió transformar columnas sin perder el punto de partida.

Otro error posible es convertir columnas que no necesitan conversión. Por ejemplo, `Transaction ID` contiene identificadores. Aunque tenga números dentro del texto, no conviene tratarlo como una variable numérica si no representa una cantidad. Un identificador no se suma, no se promedia y no se interpreta como magnitud.

Algo similar ocurre con ciertas categorías codificadas. Que una columna tenga números no significa necesariamente que sea numérica en sentido analítico. Siempre debemos preguntarnos qué representa la columna.

Una rutina razonable para convertir tipos podría ser:

```text
revisar tipos actuales
identificar columnas que necesitan conversión
observar valores problemáticos
convertir en una copia del DataFrame
verificar nuevos tipos
comparar faltantes antes y después
confirmar que las columnas convertidas se comportan como esperamos
```

La conversión de tipos es una parte central de la preparación de datos. Si las columnas no tienen el tipo adecuado, muchas operaciones posteriores pueden fallar o producir resultados engañosos.

## Resumen del capítulo

En este capítulo trabajamos con la conversión de tipos de datos.

Partimos de una idea importante: no alcanza con que una columna parezca numérica o parezca una fecha. Para que Pandas pueda operar correctamente con esos valores, necesita interpretarlos con el tipo de dato adecuado.

Primero revisamos los tipos de datos con:

```python
df.dtypes
```

y con:

```python id="sh4lzh"
df.info()
```

La salida de `info()` mostró que todas las columnas del dataset habían sido cargadas como `object`. Esto era esperable para columnas como `Transaction ID`, `Item`, `Payment Method` o `Location`, pero no para columnas como `Quantity`, `Price Per Unit`, `Total Spent` o `Transaction Date`.

Después observamos las columnas que deberían ser numéricas:

```python id="1nfyih"
df[["Quantity", "Price Per Unit", "Total Spent"]].head(15)
```

y vimos que podían contener valores problemáticos como `"ERROR"` o `"UNKNOWN"`. Esa mezcla de números y textos explica por qué Pandas las cargó como `object`.

También comprobamos que una columna cargada como texto no produce un resumen numérico adecuado. Por ejemplo, `describe()` sobre `Price Per Unit` antes de la conversión genera un resumen propio de datos textuales, no estadísticas numéricas.

Luego usamos `pd.to_numeric()` para convertir valores a formato numérico. Como algunas celdas contenían valores no convertibles, usamos:

```python id="47j5p8"
pd.to_numeric(df["Price Per Unit"], errors="coerce")
```

El parámetro `errors="coerce"` permite transformar en `NaN` los valores que no pueden convertirse a número.

Más adelante aplicamos esa conversión a varias columnas:

```python id="vwgf5t"
columnas_numericas = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

for columna in columnas_numericas:
    df_convertido[columna] = pd.to_numeric(
        df_convertido[columna],
        errors="coerce"
    )
```

Después verificamos los tipos resultantes y comparamos los faltantes antes y después de la conversión. Esta comparación fue importante porque algunos valores problemáticos que antes estaban escritos como texto pasaron a ser `NaN`.

También convertimos la columna `Transaction Date` usando:

```python id="cu2qm3"
pd.to_datetime(df_convertido["Transaction Date"], errors="coerce")
```

En este caso, los valores que no pudieron interpretarse como fechas pasaron a ser `NaT`, que representa valores temporales faltantes o no interpretables.

Finalmente validamos la estructura final con `dtypes` e `info()`, confirmando que las columnas numéricas y la columna de fecha ya estaban en tipos más adecuados para el análisis.

La idea principal de este capítulo fue:

```text id="wl8mah"
Convertir tipos de datos permite que Pandas interprete correctamente qué clase de información contiene cada columna.
```

Esta conversión es necesaria para calcular, ordenar, comparar, validar y analizar datos de manera confiable.


## Próximo paso

Ya trabajamos con valores faltantes, duplicados, limpieza de textos y conversión de tipos de datos.

El siguiente paso será profundizar en fechas y datos temporales. Ahora que sabemos convertir una columna a formato de fecha, podemos aprender a extraer información útil de ella: año, mes, día, día de la semana o períodos.

También vamos a ver por qué las fechas son especialmente importantes en muchos análisis, ya que permiten estudiar evolución, estacionalidad, cambios en el tiempo y comportamiento por períodos.